In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

ROOT = Path('..') if Path('.').name == 'notebooks' else Path('.')
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_processed_data():
    path = PROCESSED_DIR / 'master_df.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing processed data file: {path}')
    df = pd.read_csv(path, parse_dates=['ds'])
    return df

In [ ]:
def add_lag_features(df, lag_months=3):
    df = df.sort_values('ds').copy()
    for lag in range(1, lag_months + 1):
        df[f'y_lag_{lag}'] = df['y'].shift(lag)
    df['price_mom_3m'] = df['y'] / df['y_lag_3'] - 1
    df['price_mom_6m'] = df['y'] / df['y_lag_6'] - 1 if 'y_lag_6' in df.columns else np.nan
    return df

In [ ]:
def build_prophet_features(df):
    df = df.copy()
    df['cap'] = df['y'].max() * 1.2
    df['floor'] = df['y'].min() * 0.8
    df['year'] = df['ds'].dt.year
    df['month'] = df['ds'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['harvest_window'] = df['harvest_window'].fillna(0).astype(int)
    df['lean_season'] = df['lean_season'].fillna(0).astype(int)
    return df

In [ ]:
def train_prophet_model(train_df, extra_regressors):
    model = Prophet(growth='linear', seasonality_mode='multiplicative', yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    model.add_country_holidays(country_name='ID')

    for reg in extra_regressors:
        model.add_regressor(reg, standardize=True)

    model.fit(train_df[['ds', 'y'] + extra_regressors])
    return model

In [ ]:
def make_forecast(model, last_ds, periods, extra_regressors_df):
    future = model.make_future_dataframe(periods=periods, freq='MS')
    future = future[future['ds'] > last_ds] if periods > 0 else future.iloc[0:0]  # keep periods only
    future = future.reset_index(drop=True)
    future = future.merge(extra_regressors_df, on='ds', how='left')
    future = future.fillna(method='ffill').fillna(0)
    forecast = model.predict(future)
    return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()

In [ ]:
df = load_processed_data()
df = add_lag_features(df)

feature_cols = [
    'production_dev_pct',
    'rainfall_dev_pct',
    'harvest_window',
    'lean_season',
    'price_mom_3m',
    'month_sin',
    'month_cos',
]

df = build_prophet_features(df)
train_df = df.dropna(subset=feature_cols + ['y']).copy()
train_df = train_df[train_df['ds'] < '2024-01-01']
holdout_df = train_df[train_df['ds'] >= '2022-01-01']

print(f'Training rows: {len(train_df)}')
print(f'Holdout rows: {len(holdout_df)}')

In [ ]:
model = train_prophet_model(train_df, feature_cols)

future_regs = df[['ds'] + feature_cols].copy()
future_regs = future_regs[future_regs['ds'] > train_df['ds'].max()]
forecast_df = make_forecast(model, train_df['ds'].max(), periods=12, extra_regressors_df=future_regs)

perspective = forecast_df.copy()
perspective['yhat_pct_change'] = (perspective['yhat'] / train_df['y'].iloc[-1] - 1) * 100
perspective.to_csv(PROCESSED_DIR / 'forecast_df.csv', index=False)

model_path = MODEL_DIR / 'ricecast_prophet_model.pkl'
model.save_model(str(model_path))
print(f'Model saved to {model_path}')

In [ ]:
cv = cross_validation(model, initial='730 days', period='180 days', horizon='365 days', parallel='threads')
perf = performance_metrics(cv)
perf.to_csv(PROCESSED_DIR / 'prophet_performance.csv', index=False)
print('Cross-validation performance:')
print(perf[['horizon', 'mape', 'rmse', 'mae']].tail())

In [ ]:
summary = {
    'train_count': len(train_df),
    'forecast_periods': len(forecast_df),
    'rmse': float(perf['rmse'].mean()),
    'mape': float(perf['mape'].mean()),
}
print(summary)